In [3]:
import numpy as np
def clone(dict):
    newdict={}
    for a in dict:
        newdict[a]=dict[a]
    return newdict

In [19]:
class game():
    bots=[]
    num_players=0
    def __init__(self,bots=[]):
        self.bots=bots
    def add_bot(self,bot):
        bot.parent_game=self
        self.bots+=[bot]
        self.num_players+=1
    def auction(self,prop,min_bid):
        bid=min_bid
        for bot in self.bots:
            newbid=bot.check_auction(prop,bid+5)
            if(not newbid == 'OUT'):
                bid=newbid
        print('Highest bit so far is $'+str(bid))
default_preferences={'brown':1,'white':1,'light blue':1,'black':1,'purple':1,'orange':1,'red':1,'pink':1,'yellow':1,'light green':1,'green':1,'blue':1,
                      'blue rail':1, 'red rail':1,'green rail':1,'brown rail':1,'utility':1}
class bot():
    name=''
    parent_game=game()
    Money=1500
    properties=[]
    chance=blank_card()
    community=blank_card()
    preference_modifiers={}#Preferences for each color set
    modifier_for_2_of_3=1
    modifier_for_3_of_3=1
    modifier_for_2_of_2=1
    modifier_for_rail_pair=1
    modifier_for_utility_pair=1
    supply_multipliers=clone(default_preferences)#Decreases when someone offers to give the bot a particular property
    demand_multipliers=clone(default_preferences)#Increases when someone asks the bot for a particular property
    stuborness=5
    jail_turn=3 #0: Just got into jail on this turn. 1: completed 1st role 2: completed 2nd role 3 or higher: out of jail.
    def __init__(self, name, preference_modifiers):
        self.name=name
        self.preference_modifiers=clone(preference_modifiers)
    def display_assets(self):
        print()
        print()
        print()
        print('Assets for bot '+self.name)
        print('Money = $'+str(self.Money))
        print()
        print('Properties:')
        for p in self.properties:
            p.display_prop()
            print()
        print()
        print('Chance Card:')
        self.chance.display_card()
        print()
        print('Community Chest Card:')
        self.community.display_card()
        print()
        if(self.jail_turn>2):
            print(self.name+' is not in jail')
        else:
            print(self.name+' has been in jail for '+str(self.jail_turn)+' turns.')
    def display_secret(self):
        print(self.preference_modifiers)
        print(self.supply_multipliers)
        print(self.demand_multipliers)
    def get_num_properties(self,color_set): #Returns the number of properties it has for that color set
        num=0
        for prop in self.properties:
            if(prop.color_set==color_set):
                num+=1
        return num
    def get_preference(self, prop): #Returns the bot's estimated value of a card that it has, modified to account for how close it is to having a monopoly
        num_props=self.get_num_properties(prop.color_set)
        max_num=prop.cards_in_set
        if(num_props==1):
            return self.preference_modifiers[prop.color_set]
        if(num_props==2 and prop.prop_type=='color card' and max_num==2):
            return self.preference_modifiers[prop.color_set]*self.modifier_for_2_of_2
        if(num_props==2 and prop.prop_type=='color card' and max_num==3):
            return self.preference_modifiers[prop.color_set]*self.modifier_for_2_of_3
        if(num_props==3 and prop.prop_type=='color card' and max_num==3):
            return self.preference_modifiers[prop.color_set]*self.modifier_for_3_of_3
        if(num_props==2 and prop.prop_type=='railroad'):
            return self.preference_modifiers[prop.color_set]*self.modifier_for_rail_pair
        if(num_props==2 and prop.prop_type=='utility'):
            return self.preference_modifiers[prop.color_set]*self.modifier_for_utility_pair
    def check_use_card(self,situation='End of turn'):#Prints 'Used' if used (plus target if applicable). Prints 'Does not use card' if not used.
        #Situations are end of turn and beginning of turn (i.e. before dice roll). Maybe it should just ignore the beginning of the turn?
        if(situation=='End of turn'):

        if(situation=='Beginning of turn'):
            
    def check_build(self,situation='End of turn'):
        monopolies=self.get_monopolies()
        newmono=[]
        for monopoly in monopolies:
            if(not ('rail' in monopoly or 'utility' in monopoly)):
                newmono+=[monopoly]
        monopolies=newmono
        max_houses=[]
        for monopoly in monopolies:
            num=0
            for prop in self.properties:
                if(prop.color_set==monopoly and prop.num_houses>num):
                    num=prop.num_houses
            max_houses+=[num]
        highest_preference=0
        highest_num_houses=0
        index=-1
        i=0
        while(i<len(monopolies)):
            if(max_houses[i]>highest_num_houses or (max_houses[i]==highest_num_houses and self.preference_modifiers[i]>highest_preference)):
                highest_num_houses=max_houses[i]
                highest_preference=self.preference_modifiers[i]
                index=i
            i+=1
        monopoly=monopolies[index]
        props=[]
        for prop in self.properties:
            if(prop.color_set==monopoly):
                props+=[prop]
        cost=prop[0].house_cost()
        risk_mod=self.risk_modifier()
        liquifiable_cash=self.get_net_worth_without_selling()
        if(highest_num_houses==0): #Correction so it doesn't think it can mortgage properties after building the first house
            s=0
            for prop in props:
                s+=prop.cost/2
            liquifiable_cash-=s
        """
        Now the bot has to decide whether to purchase another house on this monopoly, given the cost, risk modifier, and available cash.
        I'm leaning towards just putting in a minimum spare cash threshold.
        """
    def check_jail_bailout(self):
        risk_mod=self.risk_modifier() #Might be useful since it will tell us the balance of how much money it has to spend vs. how much stuff it already has.
        liquifiable_cash=self.get_net_worth_without_selling()
        """
        Still have to figure out how to decide.
        """
    def free(self):
        self.jail_turn=3
        print(self.name + ' is no longer in jail.')
    def evaluate_trade(self, trade):
        #First, check if the trade is legal:
        cost=-trade.offered_money
        i=0
        while(i<len(trade.offered_properties)):
            if(trade.offered_properties[i].mortgaged):
                cost+=trade.offered_properties[i].cost/20
            i+=1
        if(cost>self.Money):
            print('trade rejected')
            return
        for prop in trade.asked_for_properties:
            found=False
            for my_prop in self.properties:
                if(my_prop.name==prop.name):
                    found=True
            if(not found):
                print('trade rejected')
                return
        net_value=trade.offered_money
        offered_colors=[]
        offered_props=[]
        tot_cost=[]
        for prop in trade.offered_properties:
            if(prop.color_set in offered_colors):
                i=0
                while(i<len(offered_colors)):
                    if(offered_colors[i]==prop.color_set):
                        offered_props[i]+=[prop]
                        tot_cost[i]+=[prop.cost]
                    i+=1
            else:
                offered_colors+=[prop.color_set]
                offered_props+=[[prop]]
                tot_cost+=[prop.cost]
        i=0
        while(i<len(offered_colors)):
            color=offered_colors[i]
            num_have=self.get_num_properties(color)
            num_will_have=len(offered_props[i])
            max_num=offered_props[i][0].cards_in_set
            prop_type=offered_props[i][0].prop_type
            pref=self.preference_modifier[color]*self.supply_modifier(color)
            cost_offered=tot_cost[i]
            cost_owned=self.cost_of_all_in_set(color)
            add=0
            if(prop_type=='utility'):
                if(num_will_have==2):
                    if(num_have==0):
                        add=cost_offered*modifier_for_utility_pair
                    else:
                        add=(cost_owned+cost_offered)*modifier_for_utility_pair-cost_owned
                if(num_will_have==1):
                    add=cost_offered
            if(prop_type=='railroad'):
                if(num_will_have==2):
                    if(num_have==0):
                        add=cost_offered*modifier_for_railroad_pair
                    else:
                        add=(cost_owned+cost_offered)*modifier_for_railroad_pair-cost_owned
                if(num_will_have==1):
                    add=cost_offered
            if(prop_type=='color card'):
                if(max_num==2):
                    if(num_will_have==2):
                        if(num_have==0):
                            add=cost_offered*modifier_for_2_of_2
                        else:
                            add=(cost_owned+cost_offered)*modifier_for_2_of_2-cost_owned
                    if(num_will_have==1):
                        add=cost_offered
                if(max_num==3):
                    if(num_will_have==3):
                        if(num_have==0):
                            add=cost_offered*modifier_for_3_of_3
                        if(num_have==1):
                            add=(cost_offered+cost_owned)*modifier_for_3_of_3-cost_owned
                        if(num_have==2):
                            add=(cost_offered+cost_owned)*modifier_for_3_of_3-cost_owned*modifier_for_2_of_3
                    if(num_will_have==2):
                        if(num_have==0):
                            add=cost_offered*modifier_for_2_of_3
                        else:
                            add=(cost_offered+cost_owned)*modifier_for_2_of_3-cost_owned
    
                    if(num_will_have==1):
                        add=cost_offered
            net_value+=add*pref
            i+=1
        #Now, it calculates how much value it is giving up.
        asked_colors=[]
        asked_props=[]
        tot_cost=[]
        for prop in trade.asked_for_properties:
            if(prop.color_set in asked_colors):
                i=0
                while(i<len(asked_colors)):
                    if(asked_colors[i]==prop.color_set):
                        asked_props[i]+=[prop]
                        tot_cost[i]+=[prop.cost]
                    i+=1
            else:
                asked_colors+=[prop.color_set]
                asked_props+=[[prop]]
                tot_cost+=[prop.cost]
        i=0
        while(i<len(asked_colors)):
            color=asked_colors[i]
            num_have=self.get_num_properties(color)
            num_will_have=num_have-len(asked_props[i])
            max_num=offered_props[i][0].cards_in_set
            prop_type=offered_props[i][0].prop_type
            pref=self.preference_modifier[color]*self.demand_modifier(color)
            cost_asked=tot_cost[i]
            cost_owned=self.cost_of_all_in_set(color)
            add=0
            if(prop_type=='utility'):
                if(num_will_have==0):
                    if(num_have==2):
                        add=-cost_asked*modifier_for_utility_pair
                    else:
                        add=-cost_asked
                if(num_will_have==1):
                    add=(cost_owned-cost_asked)-cost_owned*modifier_for_utility_pair
            if(prop_type=='railroad'):
                if(num_will_have==0):
                    if(num_have==2):
                        add=-cost_asked*modifier_for_railroad_pair
                    else:
                        add=-cost_asked
                if(num_will_have==1):
                    add=cost_owned-cost_asked-cost_owned*modifier_for_railroad_pair
            if(prop_type=='color card'):
                if(max_num==2):
                    if(num_will_have==0):
                        if(num_have==2):
                            add=-cost_asked*modifier_for_2_of_2
                        else:
                            add=-cost_asked
                    if(num_will_have==1):
                        add=cost_owned-cost_asked-cost_owned*modifier_for_2_of_2
                if(max_num==3):
                    if(num_will_have==0):
                        if(num_have==3):
                            add=-cost_asked*modifier_for_3_of_3
                        if(num_have==2):
                            add=-cost_asked*modifier_for_2_of_3
                        if(num_have==1):
                            add=-cost_asked
                    if(num_will_have==1):
                        if(num_have==3):
                            add=cost_owned-cost_asked-cost_owned*modifier_for_3_of_3
                        else:
                            add=cost_owned-cost_asked-cost_owned*modifier_for_2_of_3
                    if(num_will_have==2):
                        add=(cost_owned-cost_asked)*modifier_for_2_of_3-cost_owned*modifier_for_3_of_3
            net_value+=add*pref
            i+=1
        risk_mod=self.risk_modifier()
        liquifiable_cash=self.net_worth_without_selling()
        """
        Now, calculate the bot's total change in absolute networth from the trade.
        """
        net_worth_change=trade.offered_money
        for prop in trade.offered_properties:
            if(not prop.mortgaged):
                net_worth_change+=prop.cost/2
            else:
                net_worth_change-=prop.cost/20
                net_value-=prop.cost/20+prop.cost/2
        for prop in trade.asked_for_properties:
            if(not prop.mortgaged):
                net_worth_change-=prop.cost/2
            else:
                net_value+=prop.cost/2
        
        """
        Here, make a decision about whether to accept based on the net_value change, net_worth_change, and risk_mod.
        """
        #Here is one attempt:
        true_value=net_value+(1-risk_mod)*net_worth_change
        if(true_value>0):
            print('Trade accepted')
            self.accept_trade(trade)
        else:
            print('Trade rejected')
        """
        Finally, modify preferences and opponent preferences based on the trade offer
        """
        #My initial idea:
        for color in offered_colors:
            self.supply_modifiers[color]-=0.05
        for color in asked_colors:
            self.demand_modifiers[color]+=0.05
    def accept_trade(self, trade):
        self.modify_money(trade.offered_money)
        for prop in trade.offered_properties:
            self.properties+=[prop]
            if(prop.mortgaged):
                self.modify_money(-prop.cost/20)
        for prop in trade.asked_for_properties:
            newprops=[]
            i=0
            while(i<len(self.properties)):
                if(prop.name!=self.properties[i].name):
                    newprops+=[self.properties[i]]
            self.properties=newprops
    def check_buy(self, prop):
        
    def check_auction(self, prop, min_bid): #Returns 'OUT' or the bid amount
        value=0
        color=prop.color_set
        prop_type=prop.prop_type
        if(prop_type=='color card'):
            num=self.get_num_properties(color)
            max_num=prop.cards_in_set
            if(num==0):
                value=self.preference_modifiers[color]*prop.cost
            if(num==1 and max_num==2):
                value=self.preference_modifiers[color]*((prop.cost+self.cost_of_all_in_set(color))*self.modifier_for_2_of_2-self.cost_of_all_in_set(color))
            if(num==1 and max_num==3):
                value=self.preference_modifiers[color]*((prop.cost+self.cost_of_all_in_set(color))*self.modifier_for_2_of_3-self.cost_of_all_in_set(color))
            if(num==2 and max_num==3):
                value=self.preference_modifiers[color]*((prop.cost+self.cost_of_all_in_set(color))*self.modifier_for_3_of_3-self.cost_of_all_in_set(color)*self.modifier_for_2_of_3)
        if(prop_type=='railroad'):
            num=self.get_num_properties(color)
            if(num==0):
                value=self.preference_modifiers[color]*prop.cost
            if(num==1):
                value=self.preference_modifiers[color]*((prop.cost+self.cost_of_all_in_set(color))*self.modifier_for_railroad_pair-self.cost_of_all_in_set(color))
        if(prop_type=='utility'):
            num=self.get_num_properties(color) #Returns 0 or 1 based on whether the bot has the other utility
            if(num==0):
                value=self.preference_modifiers[color]*prop.cost
            if(num==1):
                value=self.preference_modifiers[color]*((prop.cost+self.cost_of_all_in_set(color))*self.modifier_for_utility_pair-self.cost_of_all_in_set(color))
        #Value of card to bot is now set.
        max_bid=value*self.risk_modifier()
        if(max_bid<min_bid):
            print(self.name+' is out.')
            return 'OUT'
        else:
            bid=int(np.random.rand()*(max_bid+1-min_bid)+min_bid)
            print(self.name+' bids $'+str(bid))
            return bid
    def get_monopolies(self):#returns all the monopolies that this player has
        colors=[]
        properties=[]
        for prop in self.properties:
            if(prop.color_set in colors):
                i=0
                index=-1
                while(i<len(colors)):
                    if(colors[i]==prop.color_set):
                        index=i
                    i+=1
                properties[index]+=[prop]
            else:
                colors+=[prop.color_set]
                properties+=[[prop]]
        monopolies=[]
        i=0
        while(i<len(colors)):
            if(len(properties[i])==properties[i][0].cards_in_set):
                monopolies+=[properties[i]]
            i+=1
        return monopolies
    def risk_modifier(self):
        networth=self.networth
        total_value_to_self=self.total_value_to_self()
        """
        Returns a modifier to multiply how much money the bot is willing to spend on a given transaction, based on how much spare cash it has.
        My idea is that it should go to 0 if total_value_to_self is a lot larger than its networth, i.e. if it doesn't have much cash to liquify but
        already has a lot of useful stuff. It should probably go to 1 as networth --> total_value_to_self, indicating it mostly just has unused cash.
        """
        return networth/total_value_to_self
    def cost_of_all_in_set(self,colorset): #returns the sum of the costs of all the cards the bot has in a given color set. Useful for some calculations
        s=0
        i=0
        while(i<len(self.properties)):
            if(self.properties[i].color_set==colorset):
                s+=self.properties[i].cost
            i+=1
        return s
    def modify_money(self,change):
        newmoney=self.Money+change
        if(newmoney>=0):
            print(self.name+'\'s cash changed from $'+str(self.Money)+' to $'+str(newmoney))
            return
        #Now it mortgages its properties, starting with the one it deems least valuable
        ratios=[]
        i=0
        while(i<len(self.properties)):
            prop=self.properties[i]
            if(prop.num_houses==0 and prop.mortgaged==False):
                preference=self.get_preference(prop)
                ratios+=[preference]
            else:
                ratios+=[10**9] #10**9 is a placeholder to denote that this property cannot be mortgaged
            i+=1
        index_order=[]
        while(len(index_order)<len(ratios)):
            m=-10**9
            index=-1
            i=0
            while(i<len(ratios)):
                if(not(i in index_order) and ratios[i]>m):
                    index=i
                    m=ratios[i]
                i+=1
            index_order+=[index]
        i=0
        while(self.Money+change<0 and i<len(index_order)):
            index=index_order[len(index_order)-1-i]
            if(ratios[index]<10**9):
                self.Money+=self.properties[index].mortgage()
            i+=1
        if(self.Money+change>=0):
            self.modify_money(change)
            return
        #Now, it sells houses
        monopolies_with_houses=[]
        i=0
        while(i<len(self.properties)):
            if(self.properties[i].num_houses>0):
                monopolies_with_houses+=[self.properties[i].num_houses]
            i+=1
        monopolies_with_houses=list(set(monopolies_with_houses))
        monopolies_in_order=[]
        while(len(monopolies_in_order)<len(monopolies_with_houses)):
            m=-10**9
            monopoly=''
            i=0
            while(i<len(monopolies_with_houses)):
                if(not(monopolies_with_houses[i] in monopolies_in_order) and self.get_preference[monopolies_with_houses[i]]>m):
                    monopoly=monopolies_with_houses[i]
                    m=self.get_preference[monopolies_with_houses[i]]
                i+=1
            monopolies_in_order+=[monopoly]
        while(self.Money+change<0):
            monopoly=monopolies_in_order[-1]
            monopolies_in_order=monopolies_in_order[:-1]
            props=[]
            i=0
            while(i<len(self.properties)):
                if(self.properties[i].color_set==monopoly):
                    props+=self.properties[i]
                i+=1
            done=False
            while(self.Money+change<0 and not done):
                max_num_houses=0
                i=0
                while(i<len(props)):
                    num=props[i].num_houses
                    if(num>max_num_houses):
                        max_num_houses=num
                    i+=1
                if(max_num_houses==0):
                    done=True
                else:
                    sell_prop=''
                    cost=0
                    i=0
                    while(i<len(props)):
                        if(props[i].num_houses==max_num_houses and props[i].cost>cost):
                            sell_prop=props[i]
                        i+=1
                    Money+=props[i].sell_house()
            while(self.Money+change<0 and len(props)>0):
                index=-1
                cost=10**9
                i=0
                while(i<len(props)):
                    if(props[i].cost<cost):
                        index=i
                        cost=props[i].cost
                    i+=1
                Money+=props[i].mortgage
                props=props[:i]+props[i+1:]
        if(self.Money+change>=0):
            self.modify_money(change)
        else:
            print(self.name+' declares bankruptcy.')
    def mortgage(self, prop_name):
        prop=''
        i=0
        while(i<len(self.properties)):
            if(self.properties[i].name==prop_name):
                prop=self.properties[i]
            i+=1
        if(prop!=''):
            self.Money+=prop.mortgage
        else:
            print('Error. '+prop_name+' could not be mortgaged because '+self.name+' does not own it.')
    def get_net_worth(self): #How much total money can be liquidated
        worth=0
        worth+=self.Money
        for prop in self.properties:
            worth+=prop.num_houses*prop.house_cost*0.5
            if(not prop.mortgaged):
                worth+=prop.cost/2
        return worth
    def get_total_value_to_self(self):
        worth=0
        worth+=self.Money
        for prop in self.properties:
            worth+=prop.num_houses*prop.house_cost
            if(prop.mortgaged):
                worth+=self.get_preferences(prop)*prop.cost-prop.cost/2
            else:
                worth+=self.get_preferences(prop)*prop.cost
        return worth
    def get_net_worth_without_selling(self):
        worth=0
        worth+=self.Money
        for prop in self.properties:
            if(prop.num_houses==0 and not prop.mortgaged):
                worth+=prop.cost/2
        return worth
    def deflate_trade_modifiers():
        for key in self.supply_modifiers:
            if(self.supply_modifiers[key]<1):
                self.supply_modifiers[key]+=0.05/stuborness
        for key in self.demand_modifiers:
            if(self.demand_modifiers[key]>1):
                self.demand_modifiers[key]-=0.05/stuborness
    def process_end_of_turn(self):
        self.check_use_card()
        self.check_build()
        self.display_assets()
        self.deflate_trade_modifiers()
    def process_beginning_of_turn(self):
        self.check_use_card(situation='Beginning of turn')
        self.check_jail_bailout()

class prop():
    num_houses=0
    color_set='N/A'
    prop_type='color card' #Can be 'color card', 'railroad', or 'utility'
    cards_in_set=3
    cost=0
    house_cost=0
    mortgaged=False
    rents=[0,0,0,0,0,0]
    name=''
    def __init__(self,name,color_set,cards_in_set,cost,rents):
        self.name=name
        self.color_set=color_set
        if('rail' in color_set):
            self.prop_type='railroad'
        if('utility' in color_set):
            self.prop_type='utility'
        if(self.prop_type=='color card'):
            if(color_set in ['brown','black','pink','blue']):
                self.cards_in_set=2
            else:
                self.cards_in_set=3
        else:
            self.cards_in_set=2
        if(color_set in ['brown','white','light blue']):
            self.house_cost=50
        if(color_set in ['black','purple','orange']):
            self.house_cost=100
        if(color_set in ['red', 'pink', 'yellow']):
            self.house_cost=150
        if(color_set in ['light green', 'green', 'blue']):
            self.house_cost=200
        self.rents=rents
        if(self.prop_type=='railroad'):
            self.rents=[50,150]
    def display_prop(self):
        print('Property:'+self.name)
        if(self.mortgaged):
            print('Property IS mortgaged')
            print('Price to unmortgage: $'+round(str(cost*0.55)))
        else:
            print('Property is NOT mortgaged')
            print('Money from mortgaging: $'+round(str(cost*0.5)))
    def mortgage(self):
        if(not self.mortgaged):
            self.mortgaged=True
            print(self.name+ ' was MORTGAGED for $'+str(self.cost/2))
            return int(round(self.cost/2,3))
        return 0
    def unmortgage(self):
        if(self.mortgaged):
            self.mortgaged=False
            print(self.name+ ' was UNMORTGAGED for $'+str(1.1*self.cost/2))
            return -int(np.ceil(round((self.cost/2)*1.1,3)))
        return 0
    def build_house(self):
        if(self.num_houses<5):
            self.num_houses+=1
            print('A house was built on '+self.name+' for $'+str(self.house_cost)+'. There are now '+str(self.num_houses)+' houses here.')
            return -self.house_cost
        return 0
    def sell_house(self):
        if(self.num_houses>0):
            self.num_houses-=1
            print('A house was sold on '+self.name+' for $'+str(self.house_cost/2)+'. There are now '+str(self.num_houses)+' houses here.')
            return self.house_cost/2
        return 0
    
All_properties=[]
class card():
    card_name='Placeholder'
    action_statement='Does nothing'
    def __init__(self,name,statement):
        self.card_name=name
        self.action_statement=statement
    def display_card(self):
        print(self.card_name)
        print(self.action_statement)
def blank_card():
    return card()
def all_chance_cards=[]
def all_community_cards=[]
def give_expanded_property_list():
    
class trade_offer():
    offered_properties=[]
    #offered_cards=[]
    offered_money=0 #Negative if you are offering the bot money
    asked_for_properties=[]
    #asked_for_cards=[]
    will_give_asker_monopoly=False
    def __init__(self,off_props,off_money,asked_props,will_give_asker_monopoly):
        if(type(off_props)!=type([])):
            off_props=[off_props]
        if(type(asked_props)!=type([])):
            asked_props=[asked_props]
        self.offered_properties=off_props
        self.asked_for_properties=asked_props
        self.offered_money=off_money
        self.will_give_asker_monopoly=will_give_asker_monopoly
        

IndentationError: expected an indented block after 'if' statement on line 87 (671675567.py, line 89)